<a href="https://colab.research.google.com/github/Emana338/Saudi-Stock-Market-Data-Platform/blob/main/Saudi_Stock_Market_Platform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:

%pip -q install duckdb polars dlt[duckdb] pyarrow plotly pandera



In [13]:

import pandas as pd
import duckdb
import polars as pl
import dlt
import pyarrow
import plotly
import pandera

print("All libraries loaded successfully")


All libraries loaded successfully


In [15]:

from google.colab import files
from pathlib import Path

uploaded = files.upload()

data_dir = Path("/content/saudi_stock_data")
data_dir.mkdir(exist_ok=True)

for filename, content in uploaded.items():
    (data_dir / filename).write_bytes(content)

print("Uploaded files:")
for file in data_dir.glob("*.csv"):
    print(file.name)


Saving Tadawul_stcks_23_4.csv to Tadawul_stcks_23_4 (1).csv
Saving Tadawul_stcks.csv to Tadawul_stcks (1).csv
Uploaded files:
Tadawul_stcks.csv
Tadawul_stcks (1).csv
Tadawul_stcks_23_4 (1).csv
Tadawul_stcks_23_4.csv


In [55]:

import pandas as pd

stock_files = [
    data_dir / "Tadawul_stcks.csv",
    data_dir / "Tadawul_stcks_23_4.csv"
]

for file in stock_files:
    df = pd.read_csv(file, low_memory=False)

    print(f"\nFile: {file.name}")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")
    print(f"Column names: {df.columns.tolist()}")
    print(f"Missing values: {df.isna().sum().sum():,}")
    print("-" * 50)



File: Tadawul_stcks.csv
Rows: 593,819
Columns: 14
Column names: ['symbol', 'name', 'trading_name ', 'sectoer', 'date', 'open', 'high', 'low', 'close', 'change', 'perc_Change', 'volume_traded ', 'value_traded', 'no_trades ']
Missing values: 27,540
--------------------------------------------------

File: Tadawul_stcks_23_4.csv
Rows: 6,992
Columns: 14
Column names: ['symbol', 'name', 'trading_name ', 'sectoer', 'date', 'open', 'high', 'low', 'close', 'change', 'perc_Change', 'volume_traded ', 'value_traded', 'no_trades ']
Missing values: 486
--------------------------------------------------


In [56]:

import pandas as pd

frames = []

for file in stock_files:
    df = pd.read_csv(file, low_memory=False)
    df.columns = df.columns.str.strip().str.lower()

    df = df.rename(columns={
        "sectoer": "sector",
        "perc_change": "percent_change"
    })

    df["source_file"] = file.name
    frames.append(df)

stocks = pd.concat(frames, ignore_index=True)
stocks["date"] = pd.to_datetime(stocks["date"], errors="coerce")

print("Total rows:", len(stocks))
print("Columns:", stocks.columns.tolist())
print("Duplicate symbol/date:", stocks.duplicated(["symbol", "date"]).sum())
print("Invalid dates:", stocks["date"].isna().sum())
print("Date range:", stocks["date"].min(), "to", stocks["date"].max())


Total rows: 600811
Columns: ['symbol', 'name', 'trading_name', 'sector', 'date', 'open', 'high', 'low', 'close', 'change', 'percent_change', 'volume_traded', 'value_traded', 'no_trades', 'source_file']
Duplicate symbol/date: 240
Invalid dates: 0
Date range: 2001-12-31 00:00:00 to 2020-04-23 00:00:00


In [57]:

key_columns = ["symbol", "date"]

duplicate_rows = stocks[
    stocks.duplicated(subset=key_columns, keep=False)
].copy()

data_columns = [
    col for col in stocks.columns
    if col not in key_columns + ["source_file"]
]

conflicts = (
    duplicate_rows
    .groupby(key_columns)[data_columns]
    .nunique(dropna=False)
)

conflicting_keys = conflicts.gt(1).any(axis=1)

print("Rows with repeated keys:", len(duplicate_rows))
print("Repeated keys:", len(conflicts))
print("Keys with different values:", conflicting_keys.sum())
print("Keys with identical values:", (~conflicting_keys).sum())

display(
    duplicate_rows.sort_values(key_columns).head(10)
)



Rows with repeated keys: 480
Repeated keys: 240
Keys with different values: 0
Keys with identical values: 240


,symbol,name,trading_name,sector,date,open,high,low,close,change,percent_change,volume_traded,value_traded,no_trades,source_file
380053,1030,Saudi Investment Bank,SAIB,Financials,2020-03-08,14.22,14.48,14.00,14.00,-1.32,-8.62,59190.0,850918.50,171.0,Tadawul_stcks.csv
598011,1030,Saudi Investment Bank,SAIB,Financials,2020-03-08,14.22,14.48,14.00,14.00,-1.32,-8.62,59190.0,850918.50,171.0,Tadawul_stcks_23_4.csv
380052,1030,Saudi Investment Bank,SAIB,Financials,2020-03-09,13.10,13.10,12.88,13.10,-0.90,-6.43,205510.0,2680831.88,185.0,Tadawul_stcks.csv
598010,1030,Saudi Investment Bank,SAIB,Financials,2020-03-09,13.10,13.10,12.88,13.10,-0.90,-6.43,205510.0,2680831.88,185.0,Tadawul_stcks_23_4.csv
380051,1030,Saudi Investment Bank,SAIB,Financials,2020-03-10,13.70,13.70,12.60,13.40,0.30,2.29,421031.0,5526616.68,351.0,Tadawul_stcks.csv
598009,1030,Saudi Investment Bank,SAIB,Financials,2020-03-10,13.70,13.70,12.60,13.40,0.30,2.29,421031.0,5526616.68,351.0,Tadawul_stcks_23_4.csv
380050,1030,Saudi Investment Bank,SAIB,Financials,2020-03-11,13.40,13.40,12.80,12.98,-0.42,-3.13,343527.0,4487703.30,337.0,Tadawul_stcks.csv
598008,1030,Saudi Investment Bank,SAIB,Financials,2020-03-11,13.40,13.40,12.80,12.98,-0.42,-3.13,343527.0,4487703.30,337.0,Tadawul_stcks_23_4.csv
380049,1030,Saudi Investment Bank,SAIB,Financials,2020-03-12,12.04,12.70,12.02,12.64,-0.34,-2.62,104309.0,1297563.88,163.0,Tadawul_stcks.csv
598007,1030,Saudi Investment Bank,SAIB,Financials,2020-03-12,12.04,12.70,12.02,12.64,-0.34,-2.62,104309.0,1297563.88,163.0,Tadawul_stcks_23_4.csv


In [58]:

from pathlib import Path

silver_dir = Path("/content/saudi_stock_data/silver")
silver_dir.mkdir(parents=True, exist_ok=True)

stocks_clean = (
    stocks
    .sort_values(["symbol", "date", "source_file"])
    .drop_duplicates(subset=["symbol", "date"], keep="first")
    .reset_index(drop=True)
)

stocks_clean.to_parquet(
    silver_dir / "stock_prices.parquet",
    index=False
)

print("Original rows:", len(stocks))
print("Clean rows:", len(stocks_clean))
print("Removed duplicates:", len(stocks) - len(stocks_clean))
print("Remaining duplicate keys:",
      stocks_clean.duplicated(["symbol", "date"]).sum())
print("Silver file saved successfully")


Original rows: 600811
Clean rows: 600571
Removed duplicates: 240
Remaining duplicate keys: 0
Silver file saved successfully


In [59]:

quality_report = pd.DataFrame({
    "missing_values": stocks_clean.isna().sum(),
    "missing_percent": (
        stocks_clean.isna().mean() * 100
    ).round(2)
})

print("Data Quality Report")
display(quality_report)

print("Invalid price ranges:",
      (
          (stocks_clean["high"] < stocks_clean["low"]) |
          (stocks_clean["open"] < 0) |
          (stocks_clean["close"] < 0)
      ).sum())

print("Negative trading volumes:",
      (stocks_clean["volume_traded"] < 0).sum())


Data Quality Report


,missing_values,missing_percent
symbol,0,0.00
name,0,0.00
trading_name,0,0.00
sector,0,0.00
date,0,0.00
open,6617,1.10
high,6859,1.14
low,6859,1.14
close,0,0.00
change,0,0.00


Invalid price ranges: 0
Negative trading volumes: 0


In [60]:

import dlt
import duckdb

pipeline = dlt.pipeline(
    pipeline_name="saudi_stock_pipeline",
    destination=dlt.destinations.duckdb(
        "/content/saudi_stock_data/stock_pipeline.duckdb"
    ),
    dataset_name="bronze"
)

print("Pipeline ready")


Pipeline ready


In [61]:

@dlt.resource(
    name="stock_prices",
    write_disposition="replace"
)
def stock_prices_source():
    for file in stock_files:
        for chunk in pd.read_csv(file, chunksize=10000):
            chunk.columns = chunk.columns.str.strip().str.lower()
            chunk = chunk.rename(columns={
                "sectoer": "sector",
                "perc_change": "percent_change"
            })
            chunk["source_file"] = file.name
            yield from chunk.to_dict(orient="records")


In [62]:

load_info = pipeline.run(stock_prices_source())

print(load_info)


Pipeline saudi_stock_pipeline load step finished in 1 minute and 54.52 seconds
1 load package(s) were loaded to destination duckdb and into dataset bronze
The duckdb destination used duckdb:////content/saudi_stock_data/stock_pipeline.duckdb location to store data
Load package 1793885862.7669256 is LOADED and contains no failed jobs


In [63]:

db_path = "/content/saudi_stock_data/stock_pipeline.duckdb"

with duckdb.connect(db_path, read_only=True) as con:
    result = con.execute("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT symbol) AS total_companies,
            MIN(TRY_CAST(date AS DATE)) AS first_date,
            MAX(TRY_CAST(date AS DATE)) AS last_date
        FROM bronze.stock_prices
    """).df()

display(result)


,total_rows,total_companies,first_date,last_date
0,600811,200,2001-12-31,2020-04-23


In [64]:

silver_path = "/content/saudi_stock_data/silver/stock_prices.parquet"

with duckdb.connect(db_path) as con:
    con.execute("CREATE SCHEMA IF NOT EXISTS silver")

    con.execute("""
        CREATE OR REPLACE TABLE silver.stock_prices AS
        SELECT *
        FROM read_parquet(?)
    """, [silver_path])

    result = con.execute("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT symbol) AS total_companies,
            COUNT(*) - COUNT(DISTINCT (symbol, date)) AS duplicate_keys
        FROM silver.stock_prices
    """).df()

display(result)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,total_companies,duplicate_keys
0,600571,200,0


In [65]:

with duckdb.connect(db_path) as con:
    con.execute("CREATE SCHEMA IF NOT EXISTS gold")

    con.execute("""
        CREATE OR REPLACE TABLE gold.daily_market AS
        SELECT
            date,
            COUNT(DISTINCT symbol) AS active_companies,
            SUM(volume_traded) AS total_volume,
            SUM(value_traded) AS total_value,
            SUM(no_trades) AS total_trades,
            AVG(percent_change) AS avg_percent_change
        FROM silver.stock_prices
        GROUP BY date
    """)

    con.execute("""
        CREATE OR REPLACE TABLE gold.sector_summary AS
        SELECT
            sector,
            COUNT(DISTINCT symbol) AS companies,
            COUNT(*) AS trading_records,
            SUM(volume_traded) AS total_volume,
            SUM(value_traded) AS total_value,
            AVG(percent_change) AS avg_percent_change
        FROM silver.stock_prices
        GROUP BY sector
    """)

    con.execute("""
        CREATE OR REPLACE TABLE gold.company_summary AS
        SELECT
            symbol,
            MAX(name) AS company_name,
            MAX(sector) AS sector,
            MIN(date) AS first_date,
            MAX(date) AS last_date,
            COUNT(*) AS trading_days,
            AVG(close) AS avg_close,
            SUM(volume_traded) AS total_volume,
            SUM(value_traded) AS total_value
        FROM silver.stock_prices
        GROUP BY symbol
    """)

    tables = con.execute("""
        SELECT
            table_name
        FROM information_schema.tables
        WHERE table_schema = 'gold'
        ORDER BY table_name
    """).df()

display(tables)


,table_name
0,company_summary
1,daily_market
2,sector_summary


In [26]:

with duckdb.connect(db_path, read_only=True) as con:
    result = con.execute("""
        SELECT
            'daily_market' AS table_name,
            COUNT(*) AS total_rows
        FROM gold.daily_market

        UNION ALL

        SELECT
            'sector_summary',
            COUNT(*)
        FROM gold.sector_summary

        UNION ALL

        SELECT
            'company_summary',
            COUNT(*)
        FROM gold.company_summary
    """).df()

display(result)


,table_name,total_rows
0,daily_market,4794
1,sector_summary,11
2,company_summary,200


In [66]:

import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

with duckdb.connect(db_path, read_only=True) as con:
    daily = con.execute(
        "SELECT * FROM gold.daily_market ORDER BY date"
    ).df()

    sectors = con.execute(
        "SELECT * FROM gold.sector_summary ORDER BY total_value DESC"
    ).df()

    companies = con.execute(
        "SELECT * FROM gold.company_summary"
    ).df()

print("Dashboard data loaded")


Dashboard data loaded


In [67]:

total_value = sectors["total_value"].sum()
total_volume = sectors["total_volume"].sum()

print(f"Companies: {len(companies):,}")
print(f"Sectors: {len(sectors):,}")
print(f"Trading Days: {len(daily):,}")
print(f"Historical Trading Value: {total_value:,.0f} SAR")
print(f"Historical Trading Volume: {total_volume:,.0f}")

fig = px.line(
    daily,
    x="date",
    y="total_value",
    title="Saudi Stock Market - Daily Trading Value",
    labels={
        "date": "Date",
        "total_value": "Trading Value (SAR)"
    }
)

fig.update_layout(template="plotly_white")
fig.show()


Companies: 200
Sectors: 11
Trading Days: 4,794
Historical Trading Value: 29,899,232,524,370 SAR
Historical Trading Volume: 1,001,365,050,502


In [68]:

fig = px.bar(
    sectors.sort_values("total_value"),
    x="total_value",
    y="sector",
    orientation="h",
    title="Trading Value by Sector",
    labels={
        "total_value": "Historical Trading Value (SAR)",
        "sector": "Sector"
    }
)

fig.update_layout(template="plotly_white")
fig.show()


In [69]:

import plotly.io as pio
from google.colab import output

output.enable_custom_widget_manager()
pio.renderers.default = "colab"

print("Plotly renderer ready")


Plotly renderer ready


In [70]:

sector_filter = widgets.Dropdown(
    options=["All Sectors"] + sorted(companies["sector"].dropna().unique().tolist()),
    description="Sector:"
)

output = widgets.Output()

def update_companies(change=None):
    with output:
        clear_output(wait=True)

        selected = sector_filter.value
        filtered = companies.copy()

        if selected != "All Sectors":
            filtered = filtered[filtered["sector"] == selected]

        top_companies = filtered.nlargest(15, "total_value")

        fig = px.bar(
            top_companies.sort_values("total_value"),
            x="total_value",
            y="company_name",
            orientation="h",
            title=f"Top Companies by Trading Value - {selected}"
        )

        fig.update_layout(
            template="plotly_white",
            height=550
        )
        fig.show()

sector_filter.observe(update_companies, names="value")

display(sector_filter, output)
update_companies()


Dropdown(description='Sector:', options=('All Sectors', 'Communication Services', 'Consumer Discretionary', 'C…

Output()

In [71]:

dashboard_dir = Path("/content/saudi_stock_data/dashboard")
dashboard_dir.mkdir(parents=True, exist_ok=True)

market_chart = px.line(
    daily,
    x="date",
    y="total_value",
    title="Daily Trading Value"
)

sector_chart = px.bar(
    sectors.sort_values("total_value"),
    x="total_value",
    y="sector",
    orientation="h",
    title="Trading Value by Sector"
)

market_chart.write_html(dashboard_dir / "daily_market.html")
sector_chart.write_html(dashboard_dir / "sector_analysis.html")

print("Dashboard files saved")


Dashboard files saved


In [72]:

import duckdb
import pandas as pd

with duckdb.connect(db_path, read_only=True) as con:
    tests = {
        "Bronze contains source data": """
            SELECT COUNT(*) = 600811
            FROM bronze.stock_prices
        """,
        "Silver has no duplicate keys": """
            SELECT COUNT(*) = COUNT(DISTINCT (symbol, date))
            FROM silver.stock_prices
        """,
        "Required fields are complete": """
            SELECT COUNT(*) = 0
            FROM silver.stock_prices
            WHERE symbol IS NULL OR date IS NULL OR close IS NULL
        """,
        "Price range is valid": """
            SELECT COUNT(*) = 0
            FROM silver.stock_prices
            WHERE high < low
               OR open < 0
               OR close < 0
        """,
        "Trading volume is non-negative": """
            SELECT COUNT(*) = 0
            FROM silver.stock_prices
            WHERE volume_traded < 0
        """,
        "Gold companies match Silver": """
            SELECT
                (SELECT COUNT(*) FROM gold.company_summary) =
                (SELECT COUNT(DISTINCT symbol) FROM silver.stock_prices)
        """
    }

    results = []

    for name, query in tests.items():
        passed = con.execute(query).fetchone()[0]
        results.append({
            "test": name,
            "status": "PASS" if passed else "FAIL"
        })

quality_results = pd.DataFrame(results)
display(quality_results)

print(
    f"Passed: {(quality_results['status'] == 'PASS').sum()}"
    f"/{len(quality_results)}"
)


,test,status
0,Bronze contains source data,PASS
1,Silver has no duplicate keys,PASS
2,Required fields are complete,PASS
3,Price range is valid,PASS
4,Trading volume is non-negative,PASS
5,Gold companies match Silver,PASS


Passed: 6/6


In [73]:

%pip -q install yfinance


In [74]:

import yfinance as yf
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path

ticker = "2222.SR"

latest = yf.download(
    ticker,
    period="1mo",
    interval="1d",
    auto_adjust=False,
    progress=False
)

if latest.empty:
    print("No data returned from the API")
else:
    latest.columns = [
        col[0] if isinstance(col, tuple) else col
        for col in latest.columns
    ]

    latest = latest.reset_index()
    latest.columns = latest.columns.str.lower()

    latest["symbol"] = 2222
    latest["source"] = "yahoo_finance"
    latest["fetched_at"] = datetime.now(timezone.utc).isoformat()

    refresh_dir = Path("/content/saudi_stock_data/refresh")
    refresh_dir.mkdir(parents=True, exist_ok=True)

    latest.to_parquet(
        refresh_dir / "aramco_latest.parquet",
        index=False
    )

    print("Refresh data downloaded successfully")
    print("Rows:", len(latest))
    print("Latest trading date:", latest["date"].max())

    display(latest.tail())


Refresh data downloaded successfully
Rows: 24
Latest trading date: 2026-11-05 00:00:00


,date,adj close,close,high,low,open,volume,symbol,source,fetched_at
19,2026-11-01,25.92,25.92,26.00,25.780001,25.900000,3377352,2222,yahoo_finance,2026-11-05T13:42:16.249528+00:00
20,2026-11-02,26.10,26.10,26.10,25.879999,25.920000,9431771,2222,yahoo_finance,2026-11-05T13:42:16.249528+00:00
21,2026-11-03,26.16,26.16,26.24,25.900000,26.080000,10296141,2222,yahoo_finance,2026-11-05T13:42:16.249528+00:00
22,2026-11-04,26.24,26.24,26.26,25.959999,26.160000,8736657,2222,yahoo_finance,2026-11-05T13:42:16.249528+00:00
23,2026-11-05,26.16,26.16,26.18,26.020000,26.120001,6915453,2222,yahoo_finance,2026-11-05T13:42:16.249528+00:00


In [75]:

import dlt

@dlt.resource(
    name="stock_updates",
    write_disposition="merge",
    primary_key=["symbol", "date"]
)
def stock_updates_source():
    update_file = "/content/saudi_stock_data/refresh/aramco_latest.parquet"

    df = pd.read_parquet(update_file)

    df = df.rename(columns={
        "volume": "volume_traded",
        "adj close": "adjusted_close"
    })

    df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")

    yield from df.to_dict(orient="records")


In [76]:

update_info = pipeline.run(stock_updates_source())

print(update_info)


Pipeline saudi_stock_pipeline load step finished in 0.58 seconds
1 load package(s) were loaded to destination duckdb and into dataset bronze
The duckdb destination used duckdb:////content/saudi_stock_data/stock_pipeline.duckdb location to store data
Load package 1793886146.2535417 is LOADED and contains no failed jobs


In [77]:

with duckdb.connect(db_path, read_only=True) as con:
    result = con.execute("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT (symbol, date)) AS unique_records,
            MAX(date) AS latest_date
        FROM bronze.stock_updates
    """).df()

display(result)


,total_rows,unique_records,latest_date
0,24,24,2026-11-05


In [78]:

pipeline.run(stock_updates_source())

with duckdb.connect(db_path, read_only=True) as con:
    result = con.execute("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT (symbol, date)) AS unique_records
        FROM bronze.stock_updates
    """).df()

display(result)


,total_rows,unique_records
0,24,24


In [79]:

%pip -q install "pyiceberg[sql-sqlite,pyarrow]"


In [80]:

from pyiceberg.catalog.sql import SqlCatalog
from pathlib import Path

lakehouse_dir = Path("/content/saudi_stock_data/lakehouse")
lakehouse_dir.mkdir(parents=True, exist_ok=True)

catalog = SqlCatalog(
    "saudi_stock",
    uri=f"sqlite:///{lakehouse_dir}/catalog.db",
    warehouse=f"file://{lakehouse_dir.resolve()}"
)

catalog.create_namespace_if_not_exists("market")

print("Iceberg catalog ready")


Iceberg catalog ready


In [81]:

import pyarrow as pa
import pyarrow.parquet as pq

silver_file = "/content/saudi_stock_data/silver/stock_prices.parquet"

arrow_table = pq.read_table(silver_file)

date_index = arrow_table.schema.get_field_index("date")

arrow_table = arrow_table.set_column(
    date_index,
    "date",
    arrow_table.column("date").cast(pa.timestamp("us"))
)

table_name = "market.stock_prices"

if catalog.table_exists(table_name):
    iceberg_table = catalog.load_table(table_name)
else:
    iceberg_table = catalog.create_table(
        identifier=table_name,
        schema=arrow_table.schema
    )

if iceberg_table.current_snapshot() is None:
    iceberg_table.append(arrow_table)
    print("Data loaded successfully")
else:
    print("Table already contains a snapshot; skipped append")

iceberg_table.refresh()

print("Total rows:", iceberg_table.scan().to_arrow().num_rows)
print("Snapshot ID:", iceberg_table.current_snapshot().snapshot_id)
print("Snapshot count:", len(iceberg_table.metadata.snapshots))



Table already contains a snapshot; skipped append
Total rows: 600571
Snapshot ID: 2973714458540092088
Snapshot count: 1


In [82]:

iceberg_table.refresh()

print("Table:", iceberg_table.name())
print("Current snapshot:", iceberg_table.current_snapshot().snapshot_id)
print("Snapshot count:", len(iceberg_table.metadata.snapshots))


Table: ('market', 'stock_prices')
Current snapshot: 2973714458540092088
Snapshot count: 1


In [83]:

%pip -q install dbt-core dbt-duckdb


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-health-checking 1.85.1 requires protobuf<8.0.0,>=7.35.1, but you have protobuf 6.33.6 which is incompatible.


In [84]:

import subprocess

result = subprocess.run(
    ["dbt", "--version"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)


Core:
  - installed: 1.12.6
  - latest:    1.12.6 - Up to date!

Plugins:
  - duckdb: 1.11.1 - Up to date!


13:43:01  dbt v2 is available, and is the default for new installations. This is dbt Core v1.
          New project - install v2: https://docs.getdbt.com/docs/local/install-dbt?utm_source=dbt-cli
          Existing v1 project - upgrade guide: https://docs.getdbt.com/docs/dbt-versions/dbt-upgrade/upgrading-to-v2?utm_source=dbt-cli




In [85]:

from pathlib import Path

dbt_dir = Path("/content/saudi_stock_dbt")
(dbt_dir / "models").mkdir(parents=True, exist_ok=True)

(dbt_dir / "dbt_project.yml").write_text("""
name: saudi_stock
version: '1.0.0'
config-version: 2

profile: saudi_stock

model-paths: ["models"]

models:
  saudi_stock:
    +materialized: table
    +schema: dbt
""".strip())

profiles_dir = Path("/content/.dbt")
profiles_dir.mkdir(exist_ok=True)

(profiles_dir / "profiles.yml").write_text(f"""
saudi_stock:
  target: dev
  outputs:
    dev:
      type: duckdb
      path: {db_path}
      threads: 2
""".strip())

print("dbt project created")


dbt project created


In [86]:

(dbt_dir / "models" / "annual_sector_activity.sql").write_text("""
SELECT
    EXTRACT(YEAR FROM date)::INTEGER AS trading_year,
    sector,
    COUNT(*) AS trading_records,
    COUNT(DISTINCT symbol) AS companies,
    SUM(volume_traded) AS total_volume,
    SUM(value_traded) AS total_value
FROM silver.stock_prices
GROUP BY 1, 2
""".strip())

print("dbt model created")


dbt model created


In [87]:

(dbt_dir / "models" / "schema.yml").write_text("""
version: 2

models:
  - name: annual_sector_activity
    description: Annual trading activity by sector
    columns:
      - name: trading_year
        data_tests:
          - not_null
      - name: sector
        data_tests:
          - not_null
      - name: trading_records
        data_tests:
          - not_null
      - name: total_value
        data_tests:
          - not_null
""".strip())

print("dbt tests created")


dbt tests created


In [88]:

import subprocess

result = subprocess.run(
    [
        "dbt", "build",
        "--project-dir", str(dbt_dir),
        "--profiles-dir", str(profiles_dir)
    ],
    capture_output=True,
    text=True
)

print(result.stdout[-5000:])
print(result.stderr[-1500:])
print("Exit code:", result.returncode)


13:43:24  Running with dbt=1.12.6
13:43:25  Registered adapter: duckdb=1.11.1
13:43:25  Found 1 model, 4 data tests, 511 macros
13:43:25  
13:43:25  Concurrency: 2 threads (target='dev')
13:43:25  
13:43:26  1 of 5 START sql table model main_dbt.annual_sector_activity ................... [RUN]
13:43:26  1 of 5 OK created sql table model main_dbt.annual_sector_activity .............. [OK in 0.28s]
13:43:26  2 of 5 START test not_null_annual_sector_activity_sector ....................... [RUN]
13:43:26  3 of 5 START test not_null_annual_sector_activity_total_value .................. [RUN]
13:43:26  2 of 5 PASS not_null_annual_sector_activity_sector ............................. [PASS in 0.11s]
13:43:26  3 of 5 PASS not_null_annual_sector_activity_total_value ........................ [PASS in 0.12s]
13:43:26  4 of 5 START test not_null_annual_sector_activity_trading_records .............. [RUN]
13:43:26  5 of 5 START test not_null_annual_sector_activity_trading_year ................. [RUN

In [89]:

%pip -q install dagster


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dbt-common 1.39.0 requires protobuf<7.0,>=6.0, but you have protobuf 7.36.2 which is incompatible.
dbt-adapters 1.24.5 requires protobuf<7.0,>=6.0, but you have protobuf 7.36.2 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which is incompatible.


In [91]:

import dagster as dg
import duckdb

@dg.asset
def bronze_stock_data():
    with duckdb.connect(db_path, read_only=True) as con:
        count = con.execute(
            "SELECT COUNT(*) FROM bronze.stock_prices"
        ).fetchone()[0]

    assert count > 0, "Bronze layer is empty"
    return count


@dg.asset(deps=[bronze_stock_data])
def silver_stock_data():
    with duckdb.connect(db_path, read_only=True) as con:
        result = con.execute("""
            SELECT
                COUNT(*) AS total_rows,
                COUNT(DISTINCT (symbol, date)) AS unique_rows
            FROM silver.stock_prices
        """).fetchone()

    assert result[0] == result[1], "Duplicate records found"
    return result[0]


@dg.asset(deps=[silver_stock_data])
def gold_stock_data():
    with duckdb.connect(db_path, read_only=True) as con:
        count = con.execute(
            "SELECT COUNT(*) FROM gold.company_summary"
        ).fetchone()[0]

    assert count > 0, "Gold layer is empty"
    return count


In [92]:

result = dg.materialize([
    bronze_stock_data,
    silver_stock_data,
    gold_stock_data
])

print("Dagster run:", "SUCCESS" if result.success else "FAILED")


2026-11-05 13:43:49 +0000 - dagster - DEBUG - __ephemeral_asset_job__ - 7352f835-b699-4322-accc-bb3eb3c58bc4 - 1553 - RUN_START - Started execution of run for "__ephemeral_asset_job__".
2026-11-05 13:43:49 +0000 - dagster - DEBUG - __ephemeral_asset_job__ - 7352f835-b699-4322-accc-bb3eb3c58bc4 - 1553 - ENGINE_EVENT - Executing steps in process (pid: 1553)
2026-11-05 13:43:49 +0000 - dagster - DEBUG - __ephemeral_asset_job__ - 7352f835-b699-4322-accc-bb3eb3c58bc4 - 1553 - RESOURCE_INIT_STARTED - Starting initialization of resources [io_manager].
2026-11-05 13:43:49 +0000 - dagster - DEBUG - __ephemeral_asset_job__ - 7352f835-b699-4322-accc-bb3eb3c58bc4 - 1553 - RESOURCE_INIT_SUCCESS - Finished initialization of resources [io_manager].
2026-11-05 13:43:49 +0000 - dagster - DEBUG - __ephemeral_asset_job__ - 7352f835-b699-4322-accc-bb3eb3c58bc4 - 1553 - LOGS_CAPTURED - Capturing logs for process (pid: 1553).
2026-11-05 13:43:49 +0000 - dagster - DEBUG - __ephemeral_asset_job__ - 7352f835-b

Dagster run: SUCCESS
